# Educational conversion propensity model

This notebook evaluates an identifier-free, one-row-per-session sample from Google's public GA4 ecommerce dataset. It is educational decision support only: it is not a causal analysis, a production targeting system, or a claim about campaign performance.

In [1]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))
from marketing_measurement.modeling.conversion import (
    evaluate_conversion_model,
    train_conversion_model,
)

data_path = project_root / 'data/observed/ga4_public_sample/conversion_model_sessions.json.gz'
sessions = pd.read_json(data_path)
sessions.shape, sessions['converted'].value_counts(dropna=False).to_dict()

((36211, 11), {'false': 35745, 'true': 466})

## Leakage boundary

The model can use only session-start calendar fields, device, coarse geography, new/returning status, and acquisition fields captured on the session's first observed event. The query and model exclude engagement, cart, checkout, purchase/revenue, transaction data, duration, and later page behavior. `user_group_bucket` is not a feature: it is a non-unique deterministic split key.

In [2]:
features = sessions.drop(columns='converted')
bundle = train_conversion_model(
    features=features,
    target=sessions['converted'],
    groups=sessions['user_group_bucket'],
)
report = evaluate_conversion_model(bundle, bundle.test)

{
    'selected_model': bundle.selected_model_name,
    'rows': len(sessions),
    'overall_prevalence': float(sessions['converted'].eq('true').mean()),
    'training_prevalence': bundle.training_prevalence,
    'split': bundle.split_metadata,
    'cross_validation': bundle.cv_scores,
}

{'selected_model': 'logistic_regression',
 'rows': 36211,
 'overall_prevalence': 0.01286901770180332,
 'training_prevalence': 0.01260098045984948,
 'split': {'method': 'group_shuffle_split',
  'group_column': 'user_group_bucket',
  'holdout_fraction_target': 0.2,
  'holdout_rows': 7245,
  'training_rows': 28966,
  'random_state': 20260940,
  'cv': 'GroupKFold(n_splits=5) on training data only',
  'group_overlap_count': 0,
  'date_range': {'start': '2020-11-01', 'end': '2021-01-31'}},
 'cross_validation': {'logistic_regression': {'roc_auc': 0.7124793019598974,
   'pr_auc': 0.03305590017028575,
   'brier_score': 0.012328780590555031},
  'random_forest': {'roc_auc': 0.6857891097874147,
   'pr_auc': 0.0281829292089436,
   'brier_score': 0.07570192718455315}}}

In [3]:
pd.DataFrame(report.model_metrics).T.assign(
    baseline_pr_auc=report.baseline_pr_auc,
    baseline_roc_auc=report.baseline_roc_auc,
    baseline_brier_score=report.baseline_brier_score,
)

,roc_auc,pr_auc,brier_score,baseline_pr_auc,baseline_roc_auc,baseline_brier_score
logistic_regression,0.679121,0.035661,0.013621,0.013941,0.5,0.013748
random_forest,0.667223,0.021713,0.085107,0.013941,0.5,0.013748


In [4]:
report.threshold_table, report.confusion_matrix, report.calibration_bins, report.subgroup_diagnostics

(               scenario  threshold  flagged_sessions  flagged_per_1000  \
 0                  0.05   0.050000                68          9.385783   
 1                  0.10   0.100000                 0          0.000000   
 2                  0.20   0.200000                 0          0.000000   
 3                  0.30   0.300000                 0          0.000000   
 4                  0.50   0.500000                 0          0.000000   
 5  capacity_50_per_1000   0.034407               363         50.103520   
 
    precision    recall  
 0   0.073529  0.049505  
 1   0.000000  0.000000  
 2   0.000000  0.000000  
 3   0.000000  0.000000  
 4   0.000000  0.000000  
 5   0.041322  0.148515  ,
 prediction       predicted_negative  predicted_positive
 actual                                                 
 actual_negative                6796                 348
 actual_positive                  86                  15,
    mean_predicted_probability  observed_conversion_rate
 0  

## Calibration decision

No probability calibration is applied. The untouched holdout is used only for diagnosis; it must not be reused to fit a calibrator. The Brier score and five-bin calibration evidence are reported for the selected uncalibrated model. A future model could use a separately nested, group-disjoint training validation process if calibration were justified.